In [1]:
import ee
import geemap

In [3]:
# 初始化 Earth Engine  ee.Authenticate()
ee.Initialize(project='ee-s87119205')

In [ ]:
# Description: split the tibet region into multiple tiles. 

# Study area
tb_area_fc = ee.FeatureCollection('projects/ee-s87119205/assets/hma')
tb_area = tb_area_fc.geometry()
tb_area_bound = tb_area.bounds()

print('tb_area_buf bounds:', tb_area_bound.getInfo())

tb_area_buf bounds: {'geodesic': False, 'type': 'Polygon', 'coordinates': [[[66.9999998898214, 26.75570326831435], [104.80000023869748, 26.75570326831435], [104.80000023869748, 46.000006037358766], [66.9999998898214, 46.000006037358766], [66.9999998898214, 26.75570326831435]]]}


In [ ]:
# based on the gee projection (epsg:3857)
def generate_grid_wgs84(xmin, ymin, xmax, ymax, dx, dy):
    xx = ee.List.sequence(xmin, ee.Number(xmax), dx)
    yy = ee.List.sequence(ymin, ee.Number(ymax), dy)
    
    def map_x(x):
        def map_y(y):
            x1 = ee.Number(x)
            x2 = ee.Number(x).add(ee.Number(dx))
            y1 = ee.Number(y)
            y2 = ee.Number(y).add(ee.Number(dy))
            coords = ee.List([x1, y1, x2, y2])
            rect = ee.Algorithms.GeometryConstructors.Rectangle(coords)
            return ee.Feature(rect)
        return yy.map(map_y)

    cells = xx.map(map_x).flatten()
    return ee.FeatureCollection(cells)

def generate_grid_utm(lonmin, latmin, lonmax, latmax, dx_utm, dy_utm, tile_num):
    xx = ee.List.sequence(1, tile_num, 1)
    
    def add_xtile(num, ls):
        tiles_ls = ee.List(ls)
        
        dummy_feat = ee.Feature(ee.Geometry.Point([0, 0]))
        last_feat = ee.Algorithms.If(
            ee.Number(num).eq(1), 
            dummy_feat, 
            ee.Feature(tiles_ls.get(-1))
        )
        coords = ee.Feature(last_feat).geometry().coordinates().flatten()
        
        point_ = ee.Algorithms.If(
            condition=ee.Number(num).eq(1), 
            trueCase=ee.Geometry.Point([lonmin, latmin]), 
            falseCase=ee.Algorithms.If(
                condition=ee.Number(coords.get(0)).gt(lonmax),
                trueCase=ee.Geometry.Point([lonmin, coords.get(7)]), 
                falseCase=ee.Geometry.Point([coords.get(2), coords.get(3)])
            )
        )
        
        point = ee.Geometry(point_)
        lon = ee.Number(point.coordinates().get(0))
        utm_zone = lon.divide(6).add(31).floor().byte()
        epsg_utm = ee.String('326').cat(utm_zone)
        proj_utm = ee.String('EPSG:').cat(epsg_utm) 
        
        point_utm = point.transform(proj_utm).coordinates()
        point_ur_utm = [
            ee.Number(dx_utm).add(point_utm.get(0)), 
            ee.Number(dy_utm).add(point_utm.get(1))
        ]
        point_ur = ee.Geometry.Point(point_ur_utm, proj_utm).transform('EPSG:4326')
        rect = ee.Algorithms.GeometryConstructors.Rectangle([point, point_ur])
        rect_ = ee.Feature(rect).set('proj', proj_utm)
        
        return tiles_ls.add(rect_)
    
    tiles_geo = xx.iterate(add_xtile, ee.List([]))

    def fill_gap(fea):
        fea_ = ee.Feature(fea)
        proj = ee.String(fea_.get('proj'))
        return fea_.buffer(distance=2000, proj=proj).bounds(10)

    tiles_geo_ = ee.List(tiles_geo).map(fill_gap)
    return ee.FeatureCollection(tiles_geo_.flatten())


In [6]:
# parameters
region_coord = tb_area_bound.coordinates()
region_lon_min = region_coord.flatten().get(0)
region_lat_min = region_coord.flatten().get(1)
region_lon_max = region_coord.flatten().get(4)
region_lat_max = region_coord.flatten().get(5)

print('region coordinates setup complete.')

tile_num = 1000
buffer = 10000
dx_utm = 100000         # 100 km
dy_utm = 100000
lonmin = region_lon_min
lonmax = region_lon_max
latmin = region_lat_min
latmax = region_lat_max


region coordinates setup complete.


In [7]:
# ------ 1. generate tiles
tb_tiles = generate_grid_utm(lonmin, latmin, lonmax, latmax, dx_utm, dy_utm, tile_num)
tb_tiles_wgs84 = generate_grid_wgs84(lonmin, latmin, lonmax, latmax, 1, 1)  # 1x1 degree

In [8]:
# ----- 2. remove empty grid and add region buffer

def remove_tile(fea):
    inter_ = tb_area.intersects(right=fea.geometry(), maxError=10)
    fea_1 = fea.set('intersection', inter_)
    fea_2 = fea_1.set('area', fea_1.area(maxError=10).divide(1e6))
    return fea_2

tb_tiles = tb_tiles.map(remove_tile).filter(ee.Filter.eq('intersection', True))
tb_tiles_wgs84 = tb_tiles_wgs84.map(remove_tile).filter(ee.Filter.eq('intersection', True))

def clear_intersection(fea):
    return fea.set('intersection', None)

tb_tiles = tb_tiles.map(clear_intersection)
tb_tiles_wgs84 = tb_tiles_wgs84.map(clear_intersection)

In [9]:
# ----- 3. add tile id: 1,2...

def add_tiles_id(fea, tiles_tb):
    tiles_ls = ee.List(tiles_tb)
    _id = tiles_ls.size().add(ee.Number(1))
    id_str = ee.String(_id.add(1000)).slice(1)
    return tiles_ls.add(ee.Feature(fea).set('tile_id', id_str))

tb_tiles = ee.FeatureCollection(ee.List(tb_tiles.iterate(add_tiles_id, ee.List([]))).flatten())
tb_tiles_wgs84 = ee.FeatureCollection(ee.List(tb_tiles_wgs84.iterate(add_tiles_id, ee.List([]))).flatten())

print('tb_tile processing step 3 complete.')

tb_tile processing step 3 complete.


In [10]:
# ----- 4. add buffer region for tiles
def add_buf(fea):
    proj = ee.String(fea.get('proj'))
    fea_buf = fea.buffer(distance=buffer).bounds(10)
    fea_buf_ = fea_buf.set('area', fea_buf.area(maxError=10).divide(1e6))
    return fea_buf_

tb_tiles_buf = tb_tiles.map(add_buf)

# tb_tiles_wgs84_buf = tb_tiles_wgs84.map(add_buf) 

In [11]:
# ----- Visualization using Geemap

empty = ee.Image().byte()
tb_outline = empty.paint(featureCollection=tb_area_fc, color=1, width=3)
tb_bound = empty.paint(featureCollection=ee.FeatureCollection(tb_area_bound), color=1, width=3)

# 创建交互式地图
Map = geemap.Map()
Map.setCenter(86.0, 32.0, 4)

# 添加图层
# Map.addLayer(tb_tiles, {}, 'tb_tiles')
Map.addLayer(tb_tiles_wgs84, {}, 'tb_tiles_wgs84')
Map.addLayer(tb_tiles_buf, {}, 'tb_tiles_buf')
# Map.addLayer(tb_tiles_wgs84_buf, {}, 'tb_tiles_wgs84_buf')
Map.addLayer(tb_outline, {'palette': 'FF0000'}, 'Tibet_outline')
Map.addLayer(tb_bound, {'palette': '0000FF'}, 'Tibet_bound')

# 显示地图 
Map

Map(center=[32.0, 86.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [13]:
# ----- Exports

# 导出到 Asset
task_asset = ee.batch.Export.table.toAsset(
    collection=tb_tiles_buf,
    description='hma_buf_asset_export',
    assetId='projects/ee-s87119205/assets/hma_tiles_buf_test' # change your own asset
)
# task_asset.start() # 取消注释以开始导出

# 导出到 Google Drive
task_drive = ee.batch.Export.table.toDrive(
    collection=tb_tiles_buf,
    description='hma_tiles_buf_shp',
    folder='GEE_Exports',
    fileNamePrefix='hma_tiles_buf',
    fileFormat='SHP'
)
task_drive.start() 

print("Export tasks created. Uncomment '.start()' lines to execute them.")

Export tasks created. Uncomment '.start()' lines to execute them.
